# Dataset

In [ ]:
url = "iris.data.csv"   # iris dataset url

# point to be noted: in original iris dataset, there was no column names, so i added column names manually in that dataset

# Decision Tree Process

In [200]:
# Imagine a dataset like this:- 

# Outlook	       Temperature	         Play Tennis
# Sunny	           Hot	                 No
# Sunny	           Hot	                 No
# Overcast	       Hot	                 Yes
# Rain	           Mild 	             Yes
# Rain	           Cool	                 Yes
# Rain	           Cool                	 No
# Overcast	       Cool	                 Yes
# Sunny	           Mild	                 No


# Entropy: It measures uncertainity(or impurity) in data.
#          i.e. if all outputs same then entropy -> 0 (pure)
#          if mixed, entropy > 0 (impure)
#          H(s) = -sum(p * log2(p))

# Entropy of Entire Dataset: P(yes) = 4/8 = 0.5  and  P(no) = 4/8 = 0.5
#          H(s) = - ((p(yes) * log2(p(yes)) + (p(no) * log2(p(no)))
#               = - (0.5*log2(0.5) + 0.5*log2(0.5)) = 1

# -------- split with a feature (outlook) --------
# Entropy of Sunny: p(yes) = 0/3 = 0 and p(no) = 3/3 = 1  that's why entropy = 0
# Entropy of Overcast: p(yes) = 2/2 = 1 and p(no) = 0/2 = 0  that's why entropy = 0
# Entropy of Rain: p(yes) = 2/3 = 0.66 and p(no) = 1/3 = 0.33  that's why entropy = -(2/3*log2(2/3) + 1/3*log2(1/3)) ~= 0.918

# So, weighted entropy after splitting is sum of probability of their samples * their respective entropy: 
#           H(afterSplit) = p(sunny)*H(sunny) + p(overcast)*H(overcast) + p(rain)*H(rain)  
#                         = 3/8 * 0 + 2/8 * 0 + 3/8 * 0.918   ~= 0.344

# what is information gain? --> how much uncertainity reduce after a split
#           information gain = H(Entire Dataset) - H(afterSplit) = 1 - 0.344 = 0.656
# so the information gain (outlook) is 0.656



# similarly we calculate IG for temperature too, and choose feature with highest IG

### Entropy

In [205]:
# y is basically the label vector which will be like [0, 2, 2, 1, 1, 1, 0, 0, 1]
# values : [0, 1, 2] and counts: [3, 4, 2]
# values returns unique datas in sorted order, and counts return their freq in that sorted order only
# so basically, p = [3/9, 4/9, 2/9] this is probability vector...
# now to calculate entropy basically  -p*log2(p)
# that means entropy = sum of [-(3/9)*log2(3/9), -(4/9)*log2(4/9), -(2/9)*log2(2/9)]
#                    = sum of - [(3/9)*log2(3/9), (4/9)*log2(4/9), (2/9)*log2(2/9)] 
#                    = - sum of [(3/9)*log2(3/9), (4/9)*log2(4/9), (2/9)*log2(2/9)]

import numpy as np

def entropy(y):
    values, counts = np.unique(y, return_counts = True)
    p = counts/len(y)    # calculating probs
    return -np.sum(p * np.log2(p))

### Split - provided the feature and threshold

In [206]:
# split the dataset in terms of feature
# X --> features, y --> labels, feature --> (here, column) which column index wanna split, threshold --> on what basis I should split...
# so we dividing some rows which are less than or equal to threshold in the feature column in left side and same with right side... 
# it creates a 2 part division, for both features and labels, so basically x will also have left side, right side and y will too

def split(x, y, feature, threshold):
    left_mask = x[ : , feature] <= threshold
    right_mask = x[ : , feature] > threshold

    return x[left_mask], x[right_mask], y[left_mask], y[right_mask]

### Calculating Information Gain - of any types of split

In [213]:
# After splitting the dataset into left and right half, we have a different purity in left half and different purity in right half
# For example: y = [1,1,1,0,0,0,0,1,1]
# after splitting: y_left = [1,1,1,0] and y_right = [1,1,0,0,0]
# so the IG = entropy(y) - (4/9*entropy(y_left) + 5/9*entropy(y_right))

def information_gain(y, y_left, y_right):
    H = entropy(y)
    HL = entropy(y_left)
    HR = entropy(y_right)

    n = len(y)
    nL = len(y_left)
    nR = len(y_right)

    HChild = ((nL/n) * HL) + ((nR/n) * HR)

    return H - HChild

### Findind Best Split 

Basically, finding best feature and best threshold, by calculating all the information gain of all the splits based on all the features and thresholds possible

In [208]:
# Now we will make the best split by trying every values of a specific feature as their threshold, and will run it for every feature

def best_split(x, y):
    best_gain = -1
    best_feature = None
    best_threshold = None

    n_feature = x.shape[1]   # shape returns (rows, cols) dimension, so for no of cols access, indexed with [1]

    for feature in range(n_feature):      # will go through each feature (column) one by one
        thresholds = np.unique(x[ : , feature])      # threshold array -> it have every value in that column

        for t in thresholds:           # making every values as a threshold once to check the IG
            x_L, x_R, y_L, y_R = split (x, y, feature, t)      # First splitting

            if len(y_L) == 0 or len(y_R) == 0 :
                continue

            gain = information_gain(y, y_L, y_R)             # noting the IG 

            if gain > best_gain:
                best_gain = gain
                best_feature = feature
                best_threshold = t

    return best_feature, best_threshold

### Defining Decision Tree's Node

In [209]:
# Now will create the decision tree, for that need one basic structure for the nodes in the tree...
# There will 2 types of node:- leaf node and decision node
# Leaf Node will only have value
# Decision Node will have 'feature' column on what basis it will divide to left or right, 'threshold', 'left' and 'right' node...

class Node:
    def __init__(self, feature = None, threshold = None, left = None, right = None, value = None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

# This is one Node Module

### Building the Decision Tree

In [210]:
# building the tree
# np.bincount(y) function, it takes an array and return the counting of all the numbers, and return as like index of output array is the number
# and its count is the value on that index. for example: y:[1, 2, 3, 2, 1, 3, 3], then bincount will return [0,2,3,3] where index 1 have 2 that means 
# 1 have appeared 2 times like that... a
# if i use .argmax() function after that, it will return the index with most count, that means np.bincount(y).argmax() will return the most occuring 
# values in that y array... 

def build_tree(X, y, depth = 0, max_depth = 3):
    if len(np.unique(y)) == 1 or depth == max_depth :
        return Node(value=np.bincount(y).argmax())

    feature, threshold = best_split(X, y)

    X_L, X_R, y_L, y_R = split(X, y, feature, threshold)

    left = build_tree(X_L, y_L, depth+1, max_depth)
    right = build_tree(X_R, y_R, depth+1, max_depth)

    return Node(feature, threshold, left, right)

### Prediction - one test sample and multiple test samples with the decision trees

In [211]:
# Lets make the prediction step now

# 1. x is one test sample, that means one row of feature to test

def predict_one(x, node):          # x is the test sample, node is the decision tree
    if node.value is not None:
        return node.value

    if x[node.feature] <= node.threshold:
        return predict_one(x, node.left)
    else:
        return predict_one(x, node.right)

# 2. X is many rows of sample... this will return an array of prediction of every test sample

def predict(X, tree):
    return np.array([predict_one(x, tree) for x in X])

# Input

### Forming the dataframe

In [201]:
import pandas as pd

df = pd.read_csv(url)
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [202]:
df.info()
df.describe()
df["species"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
 4   species       150 non-null    object 
dtypes: float64(4), object(1)
memory usage: 6.0+ KB


species
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64

### Preprocessing the labels

In [203]:
# Converting Labels to numbers

mapping = {
    "Iris-setosa": 0,
    "Iris-versicolor": 1,
    "Iris-virginica": 2
}

df["species"] = df["species"].map(mapping)

### Splitting feature and target

In [204]:
# split feature and target

X = df.iloc[ : , : -1].values   # feature
y = df.iloc[ : , -1 ].values    # labels

### Training and Testing

In [212]:
# Train and Test

tree = build_tree(X, y, max_depth=3)

predictions = predict(X, tree)

print(predictions)

accuracy = np.mean(predictions == y)
print("Accuracy:", accuracy)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 1 1 1
 1 1 1 2 1 1 1 1 1 2 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 1 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2]
Accuracy: 0.9733333333333334
